# Chunking Strategy Experiments

Compare `fixed`, `section`, and `semantic` chunking on real resume(s).
Goal: pick the strategy with cleanest, most retrieval-friendly chunks — record findings in `README.md` → Key Learnings.

Run from repo root: `jupyter notebook notebooks/chunking_experiments.ipynb`

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from app.ingestion.parser import parse_directory
from app.ingestion.chunker import chunk_document
from app.ingestion.metadata_extractor import enrich_chunks
from app.embeddings.embedder import Embedder
from app.config import RESUME_DIR

docs = parse_directory(RESUME_DIR, "resume")
print(f"Loaded {len(docs)} document(s)")
doc = docs[0]

## 1. Run all three strategies

In [ ]:
embedder = Embedder()  # needed for semantic strategy

fixed_chunks = chunk_document(doc, strategy="fixed")
section_chunks = chunk_document(doc, strategy="section")
semantic_chunks = chunk_document(doc, strategy="semantic", embedder=embedder)

for name, chunks in [("fixed", fixed_chunks), ("section", section_chunks), ("semantic", semantic_chunks)]:
    print(f"{name}: {len(chunks)} chunks")

## 2. Inspect chunk boundaries side by side

In [ ]:
def preview(chunks, n=5):
    for c in chunks[:n]:
        print(f"--- {c.chunk_id} [{c.section}] ({len(c.text)} chars) ---")
        print(c.text[:200].replace("\n", " "))
        print()

print("=== FIXED ===")
preview(fixed_chunks)

In [ ]:
print("=== SECTION ===")
preview(section_chunks)

In [ ]:
print("=== SEMANTIC ===")
preview(semantic_chunks)

## 3. Chunk size distribution

In [ ]:
import pandas as pd

rows = []
for name, chunks in [("fixed", fixed_chunks), ("section", section_chunks), ("semantic", semantic_chunks)]:
    for c in chunks:
        rows.append({"strategy": name, "chars": len(c.text)})

df = pd.DataFrame(rows)
df.groupby("strategy")["chars"].describe()

## 4. Spot-check retrieval quality per strategy

Build a quick FAISS index per strategy and compare top results for the same query.
This is a manual eyeball check — the real numbers come from `app/eval/run_eval.py`.

In [ ]:
from app.retrieval.vector_store import VectorStore
from app.embeddings.cache import embed_with_cache

query = "RAG pipeline experience with LangChain"
query_vec = embedder.embed_query(query)

for name, chunks in [("fixed", fixed_chunks), ("section", section_chunks), ("semantic", semantic_chunks)]:
    enriched = enrich_chunks(chunks)
    texts = [c.text for c in enriched]
    embeddings = embed_with_cache(embedder, texts)

    store = VectorStore()
    store.build(enriched, embeddings)
    results = store.search(query_vec, top_k=3)

    print(f"=== {name} ===")
    for chunk, score in results:
        print(f"{score:.3f}  {chunk.text[:100]}...")
    print()

## 5. Findings

_Fill in after running the above:_
- Which strategy produced the cleanest, most self-contained chunks?
- Did any strategy split relevant info across chunks (hurting retrieval)?
- Chosen default strategy for `config.CHUNK_STRATEGY`: ___

Copy conclusions into `README.md` → Key Learnings / Notes.